<a href="https://colab.research.google.com/github/eli576/USFQ_Python/blob/main/Taller_CC_Deber_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install deap
from deap import creator, base, tools
import random
import matplotlib.pyplot as plt
import pandas as pd
import time
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 7.7 MB/s eta 0:00:00


In [ ]:
#Problem parameters

#el tamaño de la población (por ejemplo: 30, 60, 120, 200)
#la probabilidad de mutación (0.05, 0.1, 0.2, 0.4)
#la probabilidad de cruce (0.6, 0.75, 0.9)
#el método de selección (torneo, ruleta, rango)
#el tipo de cruce (PMX, OX, uniforme)
#el número máximo de generaciones (200, 400, 800, 1000)


8-queens problem

In [ ]:
#Parameters
#n = 8

chrom_length = 8

population_size = 30
p_crossover = 0.6
p_mutation = 0.05
selection_method = 'torneo'
crossover_type = 'PMX'
max_generations = 200

#set random seed for reproducibility so numbers generated are the same in every try
random_seed = 42
random.seed(random_seed)


#population_size = 120
#p_crossover = 0.9
#p_mutation = 0.1
#selection_method = 'torneo'
#crossover_type = 'PMX'
#max_generations = 400


In [ ]:
#initialising for creation of 8-bit chromosome
toolbox = base.Toolbox()
toolbox.register("one_to_eight", random.randint, 1, 8)

#define fitness function strategy - minimizing conflicts
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

#create individuals of population
creator.create("Individual", list, fitness=creator.FitnessMin)

#use class previously defined to create each chromosome
toolbox.register("individualCreator", tools.initRepeat, creator.Individual, toolbox.one_to_eight, chrom_length)

#create population
toolbox.register("populationCreator", tools.initRepeat, list, toolbox.individualCreator)

In [ ]:
#fitness function
#def eight_queen_fitness(individual):
  #?????

def eight_queen_fitness(individual):
    """
    Calculate the number of attacking queen pairs for a given individual (board configuration).
    A lower number of attacking pairs corresponds to higher fitness.

    Args:
        individual (list): A list representing the queen placement on the board.
                           The index is the column, and the value is the row.

    Returns:
        tuple: A single-element tuple containing the total number of conflicts.
    """
    conflicts = 0
    n = len(individual)

    # Iterate through all possible pairs of queens
    for i in range(n):
        for j in range(i + 1, n):
            row_i = individual[i]
            col_i = i
            row_j = individual[j]
            col_j = j

            # Check for queens in the same row
            if row_i == row_j:
                conflicts += 1
            # Check for queens in the same diagonal
            elif abs(row_i - row_j) == abs(col_i - col_j):
                conflicts += 1
    # The problem asks for minimization, so a lower conflict count is better (higher fitness for minimization by DEAP's convention of -1.0 weight for FitnessMin)
    return (conflicts,)


#define fitness operator
toolbox.register("evaluate", eight_queen_fitness)

print("Fitness function 'eight_queen_fitness' defined and registered with toolbox.")




In [ ]:
#selection, crossover and mutation
#vary methods

toolbox.register("select", tools.selTournament, tournsize= 2)
toolbox.register("mate", tools.cxOnePoint)
#correct format for 8 bit?
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=p_mutation)

#toolbox.register("select", tools.selTournament, tournsize=3)
#toolbox.register("mate", tools.cxOrdered)
#toolbox.register("mate", tools.cxPartialyMatched)
#toolbox.register("mutate", tools.mutShuffleIndexes, indpb=p_mutation)
#toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.1)

In [ ]:
#create original population
population = toolbox.populationCreator (n=population_size)
generationCounter = 0

#calculate fitness of each individual
#run algorithm??
fitnessValues = list(map(toolbox.evaluate, population))

In [ ]:
#create dataframe to save results
import numpy as np

# 1. Initialize an empty list called hall_of_fame to store unique solutions (individuals with zero conflicts).
hall_of_fame = []
hall_of_fame_set = set() # Using a set for efficient uniqueness checking

# 6. Initialize stats object to store performance metrics
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)

logbook = tools.Logbook()
logbook.header = ["gen", "evals"] + stats.fields




"""
import pandas as pd
import time
import numpy as np

# 2. Define N, the size of the chessboard
N = 8

# 3. Define lists or dictionaries of hyperparameters to iterate over
POP_SIZES = [30, 60, 120, 200]
MUT_PROBS = [0.05, 0.1, 0.2, 0.4]
CX_PROBS = [0.6, 0.75, 0.9]
MAX_GENERATIONS = [200, 400, 800, 1000]

CROSSOVER_OPERATORS = {
    "cxPartialyMatched": tools.cxPartialyMatched,
    "cxOrdered": tools.cxOrdered
}

SELECTION_OPERATORS = {
    "selTournament": tools.selTournament,
    "selRoulette": tools.selRoulette
}

# 4. Initialize an empty list to store the results
results_list = []

# 5. Set N_RUNS for repeating each configuration
N_RUNS = 3

print("Libraries imported, N defined, hyperparameters and results list initialized.")
"""

In [20]:
# 4. Assign the calculated fitness values to each individual
for individual, fitness in zip(population, fitnessValues):
    individual.fitness.values = fitness

# 5. Iterate through the initial population and add any solution to hall_of_fame
for ind in population:
    if ind.fitness.values == (0.0,) and tuple(ind) not in hall_of_fame_set:
        hall_of_fame.append(ind)
        hall_of_fame_set.add(tuple(ind))

# Record initial generation statistics
record = stats.compile(population)
logbook.record(gen=0, evals=len(population), **record)
print(logbook.stream)

# 7. Loop for max_generations
for gen in range(1, max_generations + 1):
    # a. Select individuals for the next generation
    offspring = toolbox.select(population, len(population))

    # b. Clone the selected individuals to create a working copy
    offspring = list(map(toolbox.clone, offspring))

    # c. Apply crossover to pairs of individuals
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < p_crossover:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # d. Apply mutation to individuals
    for mutant in offspring:
        if random.random() < p_mutation:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # e. Evaluate the fitness of new individuals whose fitness hasn't been evaluated yet
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox.evaluate, invalid_ind)

    # f. Assign the new fitness values to the modified individuals
    for individual, fitness in zip(invalid_ind, fitnesses):
        individual.fitness.values = fitness

    # g. Replace the current population with the new generation
    population[:] = offspring

    # h. For any new individual with a fitness of 0.0, add it to the hall_of_fame if it's not already present.
    for ind in invalid_ind:
        if ind.fitness.values == (0.0,) and tuple(ind) not in hall_of_fame_set:
            hall_of_fame.append(ind)
            hall_of_fame_set.add(tuple(ind))

    # i. Calculate and record statistics for the current generation
    record = stats.compile(population)
    logbook.record(gen=gen, evals=len(invalid_ind), **record)
    print(logbook.stream)

print(f"Genetic algorithm finished after {gen} generations.")
print(f"Total unique solutions found: {len(hall_of_fame)}")
if len(hall_of_fame) > 0:
    print("First solution found:", hall_of_fame[0])


gen	evals	avg    	min	max
0  	120  	5.14167	0  	16 
1  	118  	4.04167	0  	10 
2  	112  	3.7    	0  	10 
3  	111  	3.325  	0  	10 
4  	104  	2.86667	0  	10 
5  	112  	2.725  	0  	11 
6  	110  	2.96667	0  	7  
7  	113  	3.05   	0  	9  
8  	112  	3.03333	0  	8  
9  	109  	2.86667	0  	9  
10 	113  	2.93333	0  	10 
11 	108  	2.98333	1  	11 
12 	110  	2.84167	0  	8  
13 	110  	2.64167	1  	8  
14 	107  	2.79167	1  	10 
15 	111  	2.65833	0  	9  
16 	107  	2.76667	0  	14 
17 	114  	2.65   	0  	9  
18 	115  	2.68333	0  	11 
19 	114  	2.84167	0  	9  
20 	106  	2.65833	0  	8  
21 	115  	3.075  	0  	8  
22 	111  	2.90833	0  	10 
23 	109  	2.74167	0  	8  
24 	111  	2.45833	0  	8  
25 	116  	2.325  	0  	8  
26 	114  	2.25833	0  	7  
27 	108  	1.86667	0  	11 
28 	113  	0.9    	0  	6  
29 	109  	0.433333	0  	5  
30 	105  	0.208333	0  	5  
31 	114  	0.15    	0  	7  
32 	110  	0.2     	0  	5  
33 	109  	0.191667	0  	7  
34 	109  	0.35    	0  	6  
35 	107  	0.1     	0  	3  
36 	109  	0.158333	0  	4  
37 	

In [21]:
print("\n--- Unique Solutions Found ---")
if len(hall_of_fame) > 0:
    for i, solution in enumerate(hall_of_fame):
        print(f"Solution {i+1}: {list(solution)}")
else:
    print("No unique solutions found with zero conflicts.")


--- Unique Solutions Found ---
Solution 1: [4, 2, 0, 5, 7, 1, 3, 6]
Solution 2: [3, 6, 4, 1, 5, 0, 2, 7]
Solution 3: [2, 0, 6, 4, 7, 1, 3, 5]
Solution 4: [6, 2, 0, 5, 7, 4, 1, 3]
Solution 5: [5, 2, 0, 6, 4, 7, 1, 3]
Solution 6: [6, 1, 5, 2, 0, 3, 7, 4]
Solution 7: [0, 6, 4, 7, 1, 3, 5, 2]
Solution 8: [0, 5, 7, 2, 6, 3, 1, 4]
Solution 9: [2, 5, 7, 1, 3, 0, 6, 4]
Solution 10: [5, 2, 0, 7, 4, 1, 3, 6]
Solution 11: [4, 7, 3, 0, 6, 1, 5, 2]
Solution 12: [3, 0, 4, 7, 1, 6, 2, 5]
Solution 13: [1, 6, 4, 7, 0, 3, 5, 2]
Solution 14: [4, 6, 0, 3, 1, 7, 5, 2]


In [ ]:
"""for pop_size in POP_SIZES:
    for mut_prob in MUT_PROBS:
        for cx_prob in CX_PROBS:
            for max_gen in MAX_GENERATIONS:
                for crossover_name, crossover_func in CROSSOVER_OPERATORS.items():
                    for selection_name, selection_func in SELECTION_OPERATORS.items():
                        # Dynamically register mate operator
                        toolbox.register("mate", crossover_func)

                        # Dynamically register select operator
                        if selection_func == tools.selTournament:
                            toolbox.register("select", selection_func, tournsize=3)
                        else:
                            toolbox.register("select", selection_func)

                        for run in range(N_RUNS):
                            start_time = time.time()

                            # Initialize a new population for each run
                            population = toolbox.population(n=pop_size)

                            # Hall of Fame to keep track of the best individual
                            hof = tools.HallOfFame(1)

                            # Statistics for the run
                            stats = tools.Statistics(lambda ind: ind.fitness.values)
                            stats.register("min", np.min)
                            stats.register("avg", np.mean)
                            stats.register("max", np.max)

                            # Run the genetic algorithm
                            final_population, logbook = algorithms.eaSimple(
                                population, toolbox, cxpb=cx_prob, mutpb=mut_prob,
                                ngen=max_gen, stats=stats, halloffame=hof, verbose=False
                            )

                            end_time = time.time()
                            duration = end_time - start_time

                            # Retrieve best individual and its fitness
                            best_individual = hof[0] if hof else None
                            best_fitness = best_individual.fitness.values[0] if best_individual else float('inf')

                            # Find all solutions (fitness = 0) in the final population
                            solutions_found = [ind for ind in final_population if ind.fitness.values[0] == 0]
                            num_solutions_found = len(solutions_found)

                            # Count unique solutions
                            unique_solutions = set(tuple(sol) for sol in solutions_found)
                            num_unique_solutions = len(unique_solutions)

                            # Store results
                            results_list.append({
                                "N": N,
                                "pop_size": pop_size,
                                "mut_prob": mut_prob,
                                "cx_prob": cx_prob,
                                "max_generations": max_gen,
                                "crossover_type": crossover_name,
                                "selection_type": selection_name,
                                "run_number": run + 1,
                                "execution_time": duration,
                                "best_fitness": best_fitness,
                                "best_individual": list(best_individual) if best_individual else None,
                                "num_solutions_found": num_solutions_found,
                                "num_unique_solutions": num_unique_solutions
                            })

# Convert results to DataFrame
results_df = pd.DataFrame(results_list)

print("Experimentation for 8-Queens complete. Results stored in 'results_df'.")
"""